In [ ]:
!pip install urllib3 pandas selenium openpyxl

In [1]:
import os
import time
import urllib3
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import Select

os.environ['WDM_SSL_VERIFY'] = '0'
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

from openpyxl import load_workbook
ruta_excel = r"C:\Automatizacion web\Reputacionales vencidos 07.26.xlsx"

In [2]:
df = pd.read_excel(ruta_excel,sheet_name="Hoja1",dtype=str).fillna("")
# Limpia espacios accidentales en nombres de columnas
df.columns = df.columns.str.strip()


In [ ]:
driver = webdriver.Chrome()
driver.maximize_window()
driver.get("https://enel.service-now.com/me?id=me_secp_sc_item_details&sys_id=bf5ce1ce87e98a10c3c30f650cbb354d")
wait = WebDriverWait(driver, 40)
input("presiona ENTER para continuar una vez se inicie sesión")


In [ ]:
for _, fila in df.iterrows():
    rut = str(fila.get("Rut", "")).strip()
    nombre = str(fila.get("Nombre", "")).strip()
    pais = str(fila.get("País", "")).strip()
    municipio = str(fila.get("Municipio", "")).strip()
    localizacion_geografica = str(fila.get("Localización geográfica", "")).strip()

    driver.get("https://enel.service-now.com/me?id=me_secp_sc_item_details&sys_id=bf5ce1ce87e98a10c3c30f650cbb354d")

    dropdown = None

    try:
        dropdown = wait.until(EC.element_to_be_clickable(
            (By.ID, "s2id_sp_formfield_secp_kind_of_analysis")
        ))
    except:
        pass

    if dropdown is None:
        for iframe in driver.find_elements(By.TAG_NAME, "iframe"):
            driver.switch_to.default_content()
            driver.switch_to.frame(iframe)

            try:
                dropdown = wait.until(EC.element_to_be_clickable(
                    (By.ID, "s2id_sp_formfield_secp_kind_of_analysis")
                ))
                break
            except:
                continue

    ####################################
    # Tipo de Analisis
    ####################################

    dropdown.click()

    wait.until(EC.element_to_be_clickable(
        (By.XPATH, "//div[contains(text(),'Report Light')]")
    )).click()

    ####################################
    # Perimetro de Analisis
    ####################################

    dropdown = wait.until(EC.presence_of_element_located(
        (By.ID, "s2id_sp_formfield_perimetro")
    ))

    driver.execute_script(
        "arguments[0].scrollIntoView({block:'center'});",
        dropdown
    )

    time.sleep(0.3)

    driver.execute_script("""
        const el = arguments[0].querySelector('.select2-choice');
        el.dispatchEvent(new MouseEvent('mousedown', {bubbles:true}));
        el.dispatchEvent(new MouseEvent('mouseup', {bubbles:true}));
        el.dispatchEvent(new MouseEvent('click', {bubbles:true}));
    """, dropdown)

    wait.until(EC.visibility_of_element_located(
        (By.ID, "select2-drop")
    ))

    opcion = wait.until(EC.element_to_be_clickable(
        (
            By.XPATH,
            f"//div[@id='select2-drop']//div[contains(@class,'select2-result-label') and normalize-space()='{pais}']"
        )
    ))

    driver.execute_script("""
        arguments[0].scrollIntoView({block:'center'});
    """, opcion)

    time.sleep(0.2)

    driver.execute_script("""
        const el = arguments[0];
        el.dispatchEvent(new MouseEvent('mousedown', {bubbles:true}));
        el.dispatchEvent(new MouseEvent('mouseup', {bubbles:true}));
        el.dispatchEvent(new MouseEvent('click', {bubbles:true}));
    """, opcion)

    ####################################
    # Entidad jurídica
    ####################################

    wait.until(EC.element_to_be_clickable(
        (By.ID, "s2id_sp_formfield_secp_persona_fisica_giuridica")
    )).click()

    wait.until(EC.element_to_be_clickable(
        (By.XPATH, "//div[contains(text(),'Entidad jurídica')]")
    )).click()

    ####################################
    # CIF/NIF/VAT/ID DE REGISTRO
    ####################################

    input_field = wait.until(EC.element_to_be_clickable(
        (By.ID, "sp_formfield_secp_counterpart_number")
    ))

    input_field.clear()
    input_field.send_keys(rut)

    ####################################
    # Nombre Contraparte
    ####################################
    input_nombre = wait.until(EC.element_to_be_clickable(
        (By.ID, "sp_formfield_secp_counterpart_name")
    ))

    driver.execute_script(
        "arguments[0].scrollIntoView({block:'center'});",
        input_nombre
    )

    input_nombre.click()

    # ESCAPE para cerrar cualquier select2 abierto antes de PAIS
    input_nombre.send_keys(Keys.ESCAPE)

    time.sleep(0.3)

    input_nombre.clear()
    input_nombre.send_keys(nombre)
    input_nombre.send_keys(Keys.TAB)
    ####################################
    # AVAILABLE COUNTERPART
    ####################################

    try:
        wait.until(EC.element_to_be_clickable(
            (By.ID, "s2id_sp_formfield_secp_counterpart_search")
        )).click()

        primer_elemento = wait.until(lambda d: next(
            (
                el for el in d.find_elements(
                    By.XPATH,
                    "//li[contains(@class,'select2-result-selectable') "
                    "and not(contains(@class,'select2-disabled'))]"
                    "//div[contains(@class,'select2-result-label')]"
                )
                if el.is_displayed()
                and el.text.strip()
                and "Buscando" not in el.text
                and "Searching" not in el.text
                and "No se encontraron" not in el.text
            ),
            None
        ))

        primer_elemento.click()

        # cerrar dropdowns
        ActionChains(driver).send_keys(Keys.ESCAPE).perform()

        # ESPERAR A QUE TERMINE EL AJAX / RENDER
        time.sleep(2)

        # esperar que desaparezca cualquier select2 abierto
        wait.until(lambda d: len(
            d.find_elements(
                By.XPATH,
                "//div[contains(@class,'select2-drop-active')]"
            )
        ) == 0)

    except:
        print("No se encontró contraparte")

    ####################################
    # Pais
    ####################################

    wait.until(EC.element_to_be_clickable(
        (By.ID, "s2id_sp_formfield_secp_counterpart_country")
    )).click()

    wait.until(EC.element_to_be_clickable(
        (By.XPATH, "//div[contains(text(),'Chile')]")
    )).click()

    ####################################
    # Municipio
    ####################################

    wait.until(EC.element_to_be_clickable(
        (By.ID, "s2id_sp_formfield_secp_counterpart_town")
    )).click()

    search_box = wait.until(EC.element_to_be_clickable(
        (
            By.XPATH,
            "//div[@id='select2-drop' and contains(@style,'display: block')]"
            "//input[contains(@class,'select2-input')]"
        )
    ))

    search_box.click()

    time.sleep(0.2)

    search_box.send_keys(Keys.CONTROL, "a")
    search_box.send_keys(Keys.BACKSPACE)
    search_box.send_keys(municipio)

    opcion = wait.until(lambda d: next(
        (
            el for el in d.find_elements(
                By.XPATH,
                "//div[@id='select2-drop' and contains(@style,'display: block')]"
                "//li[contains(@class,'select2-result-selectable') "
                "and not(contains(@class,'select2-disabled'))]"
            )
            if el.is_displayed()
            and el.text.strip()
            and municipio.lower() in el.text.lower()
            and "Buscando" not in el.text
            and "Searching" not in el.text
            and "No se encontraron" not in el.text
        ),
        None
    ))

    driver.execute_script(
        "arguments[0].scrollIntoView({block: 'center'});",
        opcion
    )

    time.sleep(0.2)

    driver.execute_script("""
        const el = arguments[0];
        el.dispatchEvent(new MouseEvent('mousedown', {bubbles: true, cancelable: true}));
        el.dispatchEvent(new MouseEvent('mouseup', {bubbles: true, cancelable: true}));
        el.dispatchEvent(new MouseEvent('click', {bubbles: true, cancelable: true}));
    """, opcion)

    ####################################
    # Sociedad del Grupo
    ####################################

    time.sleep(0.2)

    wait.until(EC.element_to_be_clickable(
        (By.ID, "s2id_sp_formfield_secp_countropartInfo_activity")
    )).click()

    time.sleep(0.2)

    wait.until(EC.element_to_be_clickable(
        (
            By.XPATH,
            f"//div[contains(@class,'select2-result-label') and normalize-space()='{pais}']"
        )
    )).click()

    ####################################
    # Negocio
    ####################################

    busqueda = "Global Energy and Commodity Management and Chief Pricing Officer - Risk Management - Credit Risk Management"

    wait.until(EC.element_to_be_clickable(
        (By.ID, "s2id_sp_formfield_secp_countropartInfo_scope")
    )).click()

    search_box = wait.until(EC.element_to_be_clickable(
        (
            By.XPATH,
            "//div[contains(@id,'select2-drop') "
            "and not(contains(@style,'display: none'))]"
            "//input[contains(@class,'select2-input')]"
        )
    ))

    search_box.clear()
    search_box.send_keys(busqueda)

    primer_resultado = wait.until(lambda d: next(
        (
            el for el in d.find_elements(
                By.XPATH,
                "//li[contains(@class,'select2-result-selectable') "
                "and not(contains(@class,'select2-disabled'))]"
            )
            if el.is_displayed()
            and el.text.strip()
            and "Buscando" not in el.text
            and "Searching" not in el.text
            and "No se encontraron" not in el.text
        ),
        None
    ))

    driver.execute_script(
        "arguments[0].scrollIntoView({block: 'center'});",
        primer_resultado
    )

    time.sleep(0.2)

    driver.execute_script("""
        const el = arguments[0];
        el.dispatchEvent(new MouseEvent('mousedown', {bubbles: true, cancelable: true}));
        el.dispatchEvent(new MouseEvent('mouseup', {bubbles: true, cancelable: true}));
        el.dispatchEvent(new MouseEvent('click', {bubbles: true, cancelable: true}));
    """, primer_resultado)

    ####################################
    # Empresa
    ####################################

    wait.until(EC.element_to_be_clickable(
        (By.ID, "s2id_sp_formfield_secp_prj_owner_company")
    )).click()

    search_box = wait.until(EC.element_to_be_clickable(
        (By.ID, "s2id_autogen28_search")
    ))

    search_box.click()
    search_box.send_keys(Keys.CONTROL, "a")
    search_box.send_keys(Keys.BACKSPACE)
    search_box.send_keys("Enel Generación")

    primera_opcion = wait.until(lambda d: next(
        (
            el for el in d.find_elements(
                By.XPATH,
                "//ul[@id='select2-results-28']"
                "//li[contains(@class,'select2-result-selectable') "
                "and not(contains(@class,'select2-disabled'))]"
            )
            if el.is_displayed()
            and el.text.strip()
            and "enel" in el.text.lower()
            and "chile" in el.text.lower()
            and "Cargando" not in el.text
            and "Buscando" not in el.text
            and "Searching" not in el.text
        ),
        None
    ))

    driver.execute_script(
        "arguments[0].scrollIntoView({block: 'center'});",
        primera_opcion
    )

    time.sleep(0.2)

    driver.execute_script("""
        const el = arguments[0];
        el.dispatchEvent(new MouseEvent('mousedown', {bubbles: true, cancelable: true}));
        el.dispatchEvent(new MouseEvent('mouseup', {bubbles: true, cancelable: true}));
        el.dispatchEvent(new MouseEvent('click', {bubbles: true, cancelable: true}));
    """, primera_opcion)
    #####################################################
    # Enviar
    if (1==0):
        boton_enviar = wait.until(EC.element_to_be_clickable((By.XPATH,"//button[@name='submit' and contains(@class,'btn-primary')]")))
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});",boton_enviar)

        time.sleep(0.3)

        driver.execute_script("""
            const el = arguments[0];
            el.dispatchEvent(new MouseEvent('mousedown', {bubbles:true}));
            el.dispatchEvent(new MouseEvent('mouseup', {bubbles:true}));
            el.dispatchEvent(new MouseEvent('click', {bubbles:true}));
        """, boton_enviar)
        time.sleep(1)

No se encontró contraparte
No se encontró contraparte
